# Resample LIDAR to MintPy

This notebook resamples ASO GeoTIFFs onto a MintPy geocoded grid, writes the resampled rasters into `outputs/resampled/`, and then packages them into a MintPy-style HDF5 timeseries.

## Overview

Resamples ASO LIDAR GeoTIFF snow depth measurements onto MintPy InSAR grid. Produces MintPy-compatible HDF5 timeseries for comparison analysis.

**Prerequisites:**
- MintPy installed and geocoded timeseries file available
- ASO GeoTIFF files organized by year
- `snowsar` package installed

**Workflow:**
1. Configure paths (see next cell)
2. Resample LIDAR rasters to match InSAR grid
3. Build HDF5 timeseries compatible with MintPy tools

In [ ]:
from pathlib import Path

from snowsar.utils.lidar_utils import (
    build_lidar_timeseries_h5,
    read_geotiff_stack_sorted_by_date,
    resample_many_geotiffs,
)


In [ ]:
# ============================================================================
# CONFIGURATION - Update these paths for your setup
# ============================================================================

NOTEBOOK_DIR = Path.cwd() / "LIDAR_notebooks" if (Path.cwd() / "LIDAR_notebooks").exists() else Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / "outputs_2022"
RESAMPLED_DIR = OUTPUT_DIR / "resampled"
RESAMPLED_DIR.mkdir(parents=True, exist_ok=True)

# Path to MintPy geocoded timeseries (provides reference grid for resampling)
# This file defines the target coordinate system, extent, and resolution
TIMESERIES_FILE = Path("./data/mintpy/geo_timeseries_ERA5_demErr.h5")

# Directory containing ASO GeoTIFF files organized by year
ASO_INPUT_DIR = Path("./data/aso")

# Year subdirectory to process (e.g., "2022", "2023")
YEAR = "2022"

# Glob pattern to match ASO snow depth files
# Common patterns: "*snowdepth_3m*.tif" (3m resolution) or "*snowdepth_50m*.tif" (50m)
ASO_GLOB = "*snowdepth_3m*.tif"

# Output HDF5 file - will contain LIDAR measurements in MintPy format
OUTPUT_TIMESERIES_FILE = OUTPUT_DIR / "timeseries_lidar_swe.h5"

# Set to True to re-process already resampled files (slower)
OVERWRITE_RESAMPLED = False

# ============================================================================
# Validation
# ============================================================================
if not TIMESERIES_FILE.exists():
    raise FileNotFoundError(
        f"MintPy timeseries file not found: {TIMESERIES_FILE}\n"
        f"Update TIMESERIES_FILE to point to your geo_timeseries*.h5 file"
    )
if not ASO_INPUT_DIR.exists():
    raise FileNotFoundError(
        f"ASO directory not found: {ASO_INPUT_DIR}\n"
        f"Update ASO_INPUT_DIR to point to your ASO GeoTIFF directory"
    )

aso_files = sorted((ASO_INPUT_DIR / YEAR).rglob(ASO_GLOB))
if not aso_files:
    raise FileNotFoundError(
        f"No ASO rasters found matching: {(ASO_INPUT_DIR / YEAR) / ASO_GLOB}\n"
        f"Check that YEAR and ASO_GLOB are correct"
    )

print(f"Found {len(aso_files)} ASO files to process")
aso_files[:]

## Resample source rasters

In [ ]:
written_paths = resample_many_geotiffs(
    aso_files,
    TIMESERIES_FILE,
    output_dir=RESAMPLED_DIR,
    overwrite=OVERWRITE_RESAMPLED,
)
written_paths[:5]


## Inspect stack metadata

In [ ]:
date_list, stack, metadata = read_geotiff_stack_sorted_by_date(str(RESAMPLED_DIR / "resampled_*.tif"))
{
    "num_dates": len(date_list),
    "first_date": date_list[0],
    "last_date": date_list[-1],
    "stack_shape": stack.shape,
    "first_path": metadata["paths"][0],
}


## Build MintPy-style LIDAR timeseries

In [ ]:
output_path = build_lidar_timeseries_h5(
    str(RESAMPLED_DIR / "resampled_*.tif"),
    TIMESERIES_FILE,
    OUTPUT_TIMESERIES_FILE,
)
output_path
